# Lezione 7C — Esercitazione: ReAct e Context Engineering

**Corso**: Programmazione di Applicazioni Intelligenti  
**Tipo**: Esercitazione (1 ora)  
**Prerequisiti**: Notebook 7A (agenti, ReAct from scratch) e 7B (Context Engineering)

---

**Istruzioni**: completa le celle contrassegnate con `# TODO`. Non modificare le celle già compilate (setup, tool, test). Le soluzioni sono nel notebook **7D**.


In [ ]:
# === Setup (NON MODIFICARE) ===

!pip install -q openai

from openai import OpenAI
from google.colab import userdata
import json
from datetime import datetime

client = OpenAI(
    base_url="https://api.mistral.ai/v1",
    api_key=userdata.get("MISTRAL_API_KEY"),
)

MODEL = "mistral-small-latest"
print(f"Client configurato — modello: {MODEL}")


---
## Esercizio 1 — Agente ReAct from scratch con nuovi tool (20 min)

**Obiettivo**: implementare la funzione `react_agent()` — il loop ReAct.

I 3 tool sono già definiti. Il tuo compito è scrivere il loop che:
1. Chiama il modello con i tool
2. Se `finish_reason == "tool_calls"`: esegue i tool, aggiunge i risultati, ripete
3. Se `finish_reason == "stop"`: restituisce la risposta
4. Se supera `max_steps`: restituisce un messaggio di timeout

**Test**: "In che anno è nato Leonardo da Vinci e quanti anni fa è stato?"


In [ ]:
# === Tool predefiniti (NON MODIFICARE) ===

def search_wikipedia(query: str) -> str:
    """Simula una ricerca su Wikipedia."""
    knowledge = {
        "leonardo da vinci": "Leonardo di ser Piero da Vinci (1452-1519) e' stato un inventore, artista e scienziato italiano del Rinascimento. Nato a Vinci, in Toscana, il 15 aprile 1452.",
        "albert einstein": "Albert Einstein (1879-1955) e' stato un fisico teorico tedesco. Nato a Ulm il 14 marzo 1879. Premio Nobel per la fisica nel 1921.",
        "python": "Python e' un linguaggio di programmazione ad alto livello creato da Guido van Rossum e rilasciato nel 1991.",
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return f"Nessun risultato trovato per: {query}"

def get_current_date() -> str:
    """Restituisce la data corrente."""
    now = datetime.now()
    return f"Oggi e' il {now.day}/{now.month}/{now.year}"

def calculate(expression: str) -> str:
    """Calcola un'espressione matematica in modo sicuro."""
    allowed = set("0123456789+-*/.(). ")
    if not all(c in allowed for c in expression):
        return "Errore: espressione non valida"
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Errore: {e}"

print(search_wikipedia("Leonardo da Vinci"))
print(get_current_date())
print(calculate("2025 - 1452"))


In [ ]:
# === Schema JSON dei tool (NON MODIFICARE) ===

tools = [
    {
        "type": "function",
        "function": {
            "name": "search_wikipedia",
            "description": "Cerca informazioni su Wikipedia. Restituisce un breve estratto.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Termine di ricerca"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Restituisce la data corrente. Non richiede parametri.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Calcola un'espressione matematica. Supporta +, -, *, / e parentesi.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Espressione matematica (es. '2025 - 1452')"}
                },
                "required": ["expression"]
            }
        }
    }
]

tool_registry = {
    "search_wikipedia": search_wikipedia,
    "get_current_date": get_current_date,
    "calculate": calculate,
}


In [ ]:
# === TODO: Implementa il loop ReAct ===

def react_agent(question: str, tools: list, tool_registry: dict, max_steps: int = 10) -> str:
    """
    Agente ReAct minimale.

    Algoritmo:
    1. Crea la lista messages con system prompt + domanda utente
    2. In un loop (max max_steps iterazioni):
       a. Chiama client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
       b. Se finish_reason == "stop": restituisci message.content
       c. Se finish_reason == "tool_calls":
          - Aggiungi il messaggio dell'assistente a messages
          - Per ogni tool_call: esegui la funzione dal tool_registry, aggiungi il risultato
          - Stampa la trace (tool, argomenti, risultato)
    3. Se esaurisci max_steps: restituisci messaggio di timeout
    """

    # TODO: implementa qui
    pass


In [ ]:
# === Test (NON MODIFICARE) ===

risposta = react_agent(
    question="In che anno e' nato Leonardo da Vinci e quanti anni fa e' stato?",
    tools=tools,
    tool_registry=tool_registry,
    max_steps=10
)

print("\n" + "="*60)
print("RISPOSTA FINALE:")
print("="*60)
print(risposta)


---
## Esercizio 2 — Guardrail e logging strutturato (15 min)

**Obiettivo**: migliorare l'agente con logging strutturato, conteggio token, e timeout chiaro.

La funzione deve restituire un **dizionario** con: `answer`, `steps`, `total_tokens`, `log`.


In [ ]:
# === TODO: Agente ReAct con logging ===

def react_agent_with_logging(question: str, tools: list, tool_registry: dict, max_steps: int = 10) -> dict:
    """
    Restituisce:
    {
        "answer": str,
        "steps": int,
        "total_tokens": int,
        "log": [{"step": int, "finish_reason": str, "tokens_this_step": int, "tool_calls": [...]}]
    }

    Suggerimento: response.usage.total_tokens contiene i token per ogni chiamata.
    """

    # TODO: implementa qui
    pass


In [ ]:
# === Test (NON MODIFICARE) ===

result = react_agent_with_logging(
    question="Cerca informazioni su Albert Einstein e calcola quanti anni aveva quando ha vinto il Nobel.",
    tools=tools,
    tool_registry=tool_registry
)

if result:
    print(f"Risposta: {result['answer']}")
    print(f"Step totali: {result['steps']}")
    print(f"Token totali: {result['total_tokens']}")
    for entry in result['log']:
        print(f"\nStep {entry['step']} ({entry['finish_reason']}, {entry['tokens_this_step']} tok)")
        for tc in entry.get('tool_calls', []):
            print(f"  {tc['tool']}({tc['args']}) -> {tc['result']}")


---
## Esercizio 3 — Compressione del contesto (15 min)

**Obiettivo**: implementare `compress_history()` — riassumere i messaggi più vecchi quando il contesto supera una soglia. Vedi il notebook 7B per un esempio funzionante.


In [ ]:
# === Helper (NON MODIFICARE) ===

def count_tokens_approx(messages: list) -> int:
    total_chars = sum(len(m.get("content", "") or "") for m in messages)
    return total_chars // 3


In [ ]:
# === TODO: Implementa compress_history ===

def compress_history(messages: list, max_tokens: int = 2000, keep_recent: int = 4) -> list:
    """
    Algoritmo:
    1. Se count_tokens_approx(messages) <= max_tokens: restituisci invariati
    2. Separa: system (messages[0]), da_comprimere (messages[1:-keep_recent]), recenti (messages[-keep_recent:])
    3. Costruisci stringa dai messaggi da comprimere
    4. Chiedi al modello di riassumere
    5. Ricostruisci: [system, riassunto, ...recenti]
    """

    # TODO: implementa qui
    pass


In [ ]:
# === Test (NON MODIFICARE) ===

test_messages = [
    {"role": "system", "content": "Sei un assistente utile e preciso."},
    {"role": "user", "content": "Ciao, sto organizzando un viaggio in Giappone."},
    {"role": "assistant", "content": "Che bello! Quando vorresti partire e per quanto tempo?"},
    {"role": "user", "content": "Marzo, 2 settimane, budget 3000 euro."},
    {"role": "assistant", "content": "Marzo e' perfetto per i ciliegi! Ti consiglio Tokyo, Kyoto, Osaka e Hiroshima. Japan Rail Pass 14 giorni ~310 euro."},
    {"role": "user", "content": "Posso vedere il Monte Fuji a marzo?"},
    {"role": "assistant", "content": "Il Fuji e' coperto di neve a marzo. Puoi ammirarlo da Hakone."},
    {"role": "user", "content": "Ok Hakone. Ostelli o capsule hotel?"},
    {"role": "assistant", "content": "Ostelli 20-30 euro/notte, capsule 30-40. Consiglio un mix."},
    {"role": "user", "content": "Perfetto, facciamo un mix."},
    {"role": "assistant", "content": "Ottimo! Itinerario: Tokyo 4gg, Hakone 1g, Kyoto 3gg, Osaka 2gg, Hiroshima 2gg."},
    {"role": "user", "content": "Ricapitolami tutto con i costi."},
]

print(f"Messaggi: {len(test_messages)}, Token: ~{count_tokens_approx(test_messages)}")
compressed = compress_history(test_messages, max_tokens=250, keep_recent=4)
if compressed:
    print(f"Dopo: {len(compressed)} messaggi, ~{count_tokens_approx(compressed)} token")
    for i, m in enumerate(compressed):
        print(f"  [{i}] {m['role']}: {(m.get('content','') or '')[:100]}...")


---
## Esercizio 4 (bonus) — Agente con personalità diverse (10 min)

Crea `ask_with_personality(personality_prompt, question)` che chiama il modello con un system prompt personalizzato e i tool già definiti. Testa con due personalità diverse sulla stessa domanda.


In [ ]:
# === TODO: Implementa ask_with_personality ===

def ask_with_personality(personality_prompt: str, question: str) -> str:
    """
    Chiama il modello con un system prompt personalizzato e i tool gia' definiti.
    Gestisce un singolo round di tool call (se il modello ne fa).
    Restituisce la risposta finale.
    """

    # TODO: implementa qui
    pass


# === Test (decommentare quando implementato) ===
# question = "In che anno e' nato Leonardo da Vinci?"
#
# print("=== PROFESSORE ===")
# print(ask_with_personality(
#     "Sei un professore. Spiega tutto passo per passo con tono didattico.",
#     question
# ))
# print("\n=== INGEGNERE ===")
# print(ask_with_personality(
#     "Sei un ingegnere pragmatico. Rispondi in modo ultra-conciso, solo fatti.",
#     question
# ))
